In [ ]:
!pip install -q datasets langchain-chroma langchain-huggingface chromadb transformers accelerate peft bitsandbytes sentencepiece huggingface_hub sentence-transformers sacrebleu rouge-score bert-score nltk textstat

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.1 which is incompatible.


In [ ]:
import os, re, gc, zipfile, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from google.colab import files, drive

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))

CUDA: True
GPU: Tesla T4


In [ ]:
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive')
adapter_configs = list(DRIVE_ROOT.rglob('adapter_config.json'))
for i, p in enumerate(adapter_configs, 1): print(i, '->', p.parent)
if not adapter_configs: raise FileNotFoundError('No adapter_config.json found in Drive.')
ADAPTER_CHOICE = 1  
ADAPTER_PATH = adapter_configs[ADAPTER_CHOICE - 1]
assert (ADAPTER_PATH / 'adapter_config.json').exists()
assert (ADAPTER_PATH / 'adapter_model.safetensors').exists() or (ADAPTER_PATH / 'adapter_model.bin').exists()
print('Selected adapter:', ADAPTER_PATH)

Mounted at /content/drive


FileNotFoundError: No adapter_config.json found in Drive.

In [ ]:
from huggingface_hub import login, whoami
from getpass import getpass
hf_token = getpass('Paste your NEW Hugging Face read token: ')
login(token=hf_token, add_to_git_credential=False)
print('Logged in as:', whoami()['name'])
del hf_token

In [ ]:
uploaded = files.upload()
if not uploaded: raise ValueError('Upload the ChromaDB ZIP.')
zip_path = Path(next(iter(uploaded))).resolve()
extract_dir = Path('/content/radiology_kb_2k_rag')
extract_dir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_path) as zf: zf.extractall(extract_dir)
candidates = [p.parent for p in extract_dir.rglob('chroma.sqlite3')]
if not candidates: raise FileNotFoundError('No chroma.sqlite3 found in ZIP.')
CHROMA_DIRECTORY = candidates[0]
print('Chroma directory:', CHROMA_DIRECTORY)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
embedding_model = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device': DEVICE},
    encode_kwargs={'normalize_embeddings': True}
)
vector_store = Chroma(
    collection_name='biolaysumm_rrg_terms',
    persist_directory=str(CHROMA_DIRECTORY),
    embedding_function=embedding_model
)
print('Chroma documents:', vector_store._collection.count())
if vector_store._collection.count() == 0:
    raise ValueError('Collection is empty or collection_name is incorrect.')
kb = vector_store._collection.get(include=['documents','metadatas'])
KB_ENTRIES = [
    {'term': (m or {}).get('term',''), 'definition': d, 'metadata': m or {}}
    for d, m in zip(kb['documents'], kb['metadatas'])
]
print('KB entries:', len(KB_ENTRIES))

In [ ]:
EXCLUDED_RETRIEVAL_TERMS = {
    'no evidence of','without','cannot exclude','may represent','possible','probable',
    'stable','unchanged','interval increase','interval decrease','improved','resolved','new','acute'
}
def normalize_text(text):
    return re.sub(r'\s+', ' ', re.sub(r'[^a-z0-9\s-]', ' ', str(text).lower())).strip()

def term_occurs_in_report(term, report):
    term, report = normalize_text(term), normalize_text(report)
    ending = '' if term.endswith('s') else 's?'
    return bool(re.search(r'(?<![a-z0-9])' + re.escape(term) + ending + r'(?![a-z0-9])', report))

def retrieve_definitions(report, maximum_results=4, fallback_threshold=0.75):
    exact = []
    for e in KB_ENTRIES:
        term = e['term']; norm = normalize_text(term)
        if term and norm not in EXCLUDED_RETRIEVAL_TERMS and term_occurs_in_report(term, report):
            exact.append({'term':term,'definition':e['definition'],'score':1.0,'match_type':'exact'})
    exact.sort(key=lambda x: len(normalize_text(x['term'])), reverse=True)
    selected, selected_terms = [], []
    for item in exact:
        norm = normalize_text(item['term'])
        if any(norm in old for old in selected_terms): continue
        selected.append(item); selected_terms.append(norm)
        if len(selected) >= maximum_results: break
    if selected: return selected
    fallback = []
    for doc, score in vector_store.similarity_search_with_relevance_scores(report, k=5):
        term = doc.metadata.get('term','')
        if score >= fallback_threshold and term and normalize_text(term) not in EXCLUDED_RETRIEVAL_TERMS:
            fallback.append({'term':term,'definition':doc.page_content,'score':float(score),'match_type':'semantic'})
    return fallback[:2]
print('Version 1 retriever ready.')

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL_NAME = 'meta-llama/Llama-3.2-1B-Instruct'
compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=compute_dtype, bnb_4bit_use_double_quant=True
)
tokenizer = AutoTokenizer.from_pretrained(str(ADAPTER_PATH))
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'
tokenizer.clean_up_tokenization_spaces = False
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME, quantization_config=bnb, device_map='auto', torch_dtype=compute_dtype
)
model = PeftModel.from_pretrained(base_model, str(ADAPTER_PATH), is_trainable=False)
model.config.use_cache = True
model.eval()
print('Loaded adapter:', ADAPTER_PATH)

In [ ]:
def format_context(entries):
    return 'No relevant glossary definitions were retrieved.' if not entries else '\n'.join(f"- {e['definition']}" for e in entries)

def create_sentence_prompt(sentence, context):
    system = '''Simplify exactly one radiology-report sentence.
Preserve every finding, negation, uncertainty, severity, location, laterality, and comparison.
Use definitions only to explain terms present in this sentence. Do not add causes, diagnoses,
symptoms, history, treatment, recommendations, or prognosis. Air trapping means air remains
inside the lungs; it does not mean pneumonia. Pneumothorax means air around a lung; do not
confuse it with air trapping. Chronic means long-term. Dorsal kyphosis means increased outward
curvature of the upper back. Do not start with "A patient presents." Return only one simplified sentence.'''
    user = f'REPORT SENTENCE:\n{sentence}\n\nRELEVANT DEFINITIONS:\n{context}\n\nSimplified sentence:'
    return tokenizer.apply_chat_template(
        [{'role':'system','content':system},{'role':'user','content':user}],
        tokenize=False, add_generation_prompt=True
    )

def generate_one(prompt, max_new_tokens=70):
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1024)
    inputs = {k:v.to(model.device) for k,v in inputs.items()}
    with torch.inference_mode():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False,
            repetition_penalty=1.05, pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(out[0, inputs['input_ids'].shape[1]:],
                            skip_special_tokens=True, clean_up_tokenization_spaces=False).strip()

def clean_text(text):
    text = re.sub(r'^\*{0,2}(patient\s+summary|summary)\*{0,2}\s*:?\s*', '', str(text).strip(), flags=re.I)
    text = re.sub(r'^(a\s+patient\s+presents\s+with|the\s+patient\s+presents\s+with)\s+', '', text, flags=re.I)
    for wrong, right in {'pneumohorax':'pneumothorax','pneumothorace':'pneumothorax','pneumo thorax':'pneumothorax','ky phosis':'kyphosis'}.items():
        text = re.sub(r'\b'+re.escape(wrong)+r'\b', right, text, flags=re.I)
    text = re.sub(r'\s+', ' ', text).strip()
    return text + ('' if not text or text[-1] in '.!?' else '.')

def split_sentences(report):
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', str(report).strip()) if s.strip()]

def generate_rag_summary(report, show_retrieval=False):
    report = str(report).strip()
    if not report: raise ValueError('Radiology report is empty.')
    summaries, all_entries = [], []
    for sentence in split_sentences(report):
        entries = retrieve_definitions(sentence)
        all_entries.extend(entries)
        summaries.append(clean_text(generate_one(create_sentence_prompt(sentence, format_context(entries)))))
    unique, seen = [], set()
    for e in all_entries:
        key = normalize_text(e['term'])
        if key not in seen: seen.add(key); unique.append(e)
    if show_retrieval:
        for e in unique: print(f"- {e['term']} ({e['match_type']}): {e['score']:.4f}")
        print('SUMMARY:', ' '.join(x for x in summaries if x))
    return {
        'summary':' '.join(x for x in summaries if x),
        'retrieved_terms':[e['term'] for e in unique],
        'retrieval_scores':[e['score'] for e in unique],
        'match_types':[e['match_type'] for e in unique]
    }

In [ ]:
sample_report = 'The chest shows significant air trapping. Bilateral apical chronic changes are present. Dorsal kyphosis is noted. No evidence of pneumothorax.'
sample_result = generate_rag_summary(sample_report, show_retrieval=True)

In [ ]:
from datasets import load_dataset
dataset = load_dataset('BioLaySumm/BioLaySumm2025-LaymanRRG-opensource-track')
shuffled = dataset['train'].shuffle(seed=42)
evaluation_dataset = shuffled.select(range(2100, 2200))
reference_count = sum(bool(str(row.get('layman_report','')).strip()) for row in evaluation_dataset)
print('Test samples:', len(evaluation_dataset), '| nonempty references:', reference_count)
assert len(evaluation_dataset) == 100
assert reference_count == 100

In [ ]:
results = []
for number, example in enumerate(tqdm(evaluation_dataset, desc='2K Fine-tuned + RAG'), 1):
    report = str(example.get('radiology_report','')).strip()
    reference = str(example.get('layman_report','')).strip()
    try:
        result = generate_rag_summary(report)
        summary = result['summary'].strip()
        results.append({
            'sample_number':number, 'shuffled_dataset_index':2099+number,
            'source':example.get('source',''), 'radiology_report':report,
            'reference_layman_summary':reference,
            'retrieved_terms':' | '.join(result['retrieved_terms']),
            'retrieval_scores':' | '.join(f'{x:.4f}' for x in result['retrieval_scores']),
            'match_types':' | '.join(result['match_types']),
            'finetuned_rag_summary':summary,
            'error':'' if summary else 'Empty generated summary'
        })
    except Exception as e:
        results.append({
            'sample_number':number, 'shuffled_dataset_index':2099+number,
            'source':example.get('source',''), 'radiology_report':report,
            'reference_layman_summary':reference, 'retrieved_terms':'',
            'retrieval_scores':'', 'match_types':'', 'finetuned_rag_summary':'',
            'error':str(e)
        })
results_df = pd.DataFrame(results)
print('Rows:',len(results_df),'| valid summaries:',results_df['finetuned_rag_summary'].fillna('').str.strip().ne('').sum(),
      '| errors:',results_df['error'].fillna('').str.strip().ne('').sum())

In [ ]:
RAW_FILE = 'finetuned_llama_2k_rag_100_results.csv'
results_df.to_csv(RAW_FILE, index=False, encoding='utf-8-sig')
display(results_df.head())
files.download(RAW_FILE)

In [ ]:
valid = (
    results_df['reference_layman_summary'].fillna('').str.strip().ne('')
    & results_df['finetuned_rag_summary'].fillna('').str.strip().ne('')
    & results_df['error'].fillna('').str.strip().eq('')
)
eval_df = results_df.loc[valid].copy().reset_index(drop=True)
references = eval_df['reference_layman_summary'].astype(str).str.strip().tolist()
predictions = eval_df['finetuned_rag_summary'].astype(str).str.strip().tolist()
reports = eval_df['radiology_report'].astype(str).str.strip().tolist()
print('Valid evaluation pairs:',len(eval_df),'of',len(results_df))
if not references: raise ValueError('No valid prediction/reference pairs.')

In [ ]:
import sacrebleu, nltk
from rouge_score import rouge_scorer
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import wordpunct_tokenize
nltk.download('wordnet'); nltk.download('omw-1.4')
bleu_score = sacrebleu.corpus_bleu(predictions, [references]).score
rouge = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)
rs = [rouge.score(ref,pred) for ref,pred in zip(references,predictions)]
rouge1_scores = [x['rouge1'].fmeasure for x in rs]
rouge2_scores = [x['rouge2'].fmeasure for x in rs]
rougeL_scores = [x['rougeL'].fmeasure for x in rs]
meteor_scores = [meteor_score([wordpunct_tokenize(ref.lower())],wordpunct_tokenize(pred.lower())) for ref,pred in zip(references,predictions)]
print(f'BLEU: {bleu_score:.6f}')
print(f'ROUGE-1: {np.mean(rouge1_scores):.6f}')
print(f'ROUGE-2: {np.mean(rouge2_scores):.6f}')
print(f'ROUGE-L: {np.mean(rougeL_scores):.6f}')
print(f'METEOR: {np.mean(meteor_scores):.6f}')

In [ ]:
from bert_score import score
P,R,F1 = score(cands=predictions, refs=references, lang='en',
               model_type='roberta-large', device=DEVICE, batch_size=1, verbose=True)
bert_precision = P.mean().item()
bert_recall = R.mean().item()
bert_f1 = F1.mean().item()
print(f'BERTScore Precision: {bert_precision:.6f}')
print(f'BERTScore Recall: {bert_recall:.6f}')
print(f'BERTScore F1: {bert_f1:.6f}')

In [ ]:
from sentence_transformers import SentenceTransformer, util
import textstat
semantic_model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2', device=DEVICE)
ref_emb = semantic_model.encode(references, convert_to_tensor=True, normalize_embeddings=True, batch_size=16, show_progress_bar=True)
pred_emb = semantic_model.encode(predictions, convert_to_tensor=True, normalize_embeddings=True, batch_size=16, show_progress_bar=True)
semantic_scores = [util.cos_sim(pred_emb[i],ref_emb[i]).item() for i in range(len(predictions))]
fkgl_scores = [textstat.flesch_kincaid_grade(x) for x in predictions]
dale_scores = [textstat.dale_chall_readability_score(x) for x in predictions]
cli_scores = [textstat.coleman_liau_index(x) for x in predictions]
semantic_mean = float(np.mean(semantic_scores))
fkgl_mean = float(np.mean(fkgl_scores))
dale_mean = float(np.mean(dale_scores))
cli_mean = float(np.mean(cli_scores))
print(f'Semantic Similarity: {semantic_mean:.6f}')
print(f'FKGL: {fkgl_mean:.6f}')
print(f'Dale-Chall: {dale_mean:.6f}')
print(f'Coleman-Liau: {cli_mean:.6f}')

In [ ]:
for name in ['model','base_model','semantic_model','embedding_model']:
    if name in globals(): del globals()[name]
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
print('Generation and semantic evaluation finished.')

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
nli_name = 'MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli'
nli_tokenizer = AutoTokenizer.from_pretrained(nli_name)
nli_model = AutoModelForSequenceClassification.from_pretrained(nli_name).to(DEVICE).eval()

def check_nli(report, summary):
    inputs = nli_tokenizer(str(report), str(summary), return_tensors='pt',
                           truncation=True, max_length=512, padding=True)
    inputs = {k:v.to(DEVICE) for k,v in inputs.items()}
    with torch.no_grad(): probs = torch.softmax(nli_model(**inputs).logits, dim=-1)[0]
    scores = {str(nli_model.config.id2label[i]).lower():float(probs[i]) for i in range(len(probs))}
    label = max(scores, key=scores.get)
    return label, label != 'entailment', scores

nli_rows = []
for report, summary in tqdm(zip(reports,predictions), total=len(predictions), desc='NLI'):
    label, flag, scores = check_nli(report, summary)
    nli_rows.append({
        'nli_label':label, 'entailment_score':scores.get('entailment',0.0),
        'neutral_score':scores.get('neutral',0.0),
        'contradiction_score':scores.get('contradiction',0.0),
        'nli_non_entailment_flag':flag
    })
nli_df = pd.DataFrame(nli_rows)
eval_df = pd.concat([eval_df.reset_index(drop=True),nli_df],axis=1)
nli_count = int(eval_df['nli_non_entailment_flag'].sum())
nli_rate = 100*nli_count/len(eval_df)
print(f'NLI non-entailment: {nli_count}/{len(eval_df)} ({nli_rate:.2f}%)')

In [ ]:
metrics_df = pd.DataFrame({
    'Metric':['BLEU','ROUGE-1','ROUGE-2','ROUGE-L','METEOR',
              'BERTScore Precision','BERTScore Recall','BERTScore F1',
              'Semantic Similarity','FKGL','Dale-Chall','Coleman-Liau',
              'NLI-estimated non-entailment rate'],
    'Score':[bleu_score,np.mean(rouge1_scores),np.mean(rouge2_scores),
             np.mean(rougeL_scores),np.mean(meteor_scores),
             bert_precision,bert_recall,bert_f1,semantic_mean,fkgl_mean,
             dale_mean,cli_mean,f'{nli_rate:.2f}%']
})
display(metrics_df)
DETAILS_FILE = 'finetuned_llama_2k_rag_100_results_with_nli.csv'
METRICS_FILE = 'finetuned_llama_2k_rag_complete_metrics.csv'
eval_df.to_csv(DETAILS_FILE,index=False,encoding='utf-8-sig')
metrics_df.to_csv(METRICS_FILE,index=False,encoding='utf-8-sig')
files.download(DETAILS_FILE)
files.download(METRICS_FILE)